# UNIVERSAL ADAPTIVE PIPELINE: ANALISIS SENTIMEN & OPINION MINING
### COMPETITION-GRADE ARCHITECTURE (100% DOMAIN-AGNOSTIC & ZERO-HARDCODING)

**Prinsip Desain:**
1. **Auto-Schema Discovery**: Tidak ada nama kolom yang di-hardcode. Pipeline otomatis mendeteksi kolom ID, Text, dan Target jika tidak diisi secara eksplisit.
2. **Domain-Agnostic Cleaning**: Pembersih teks berbasis aturan modular struktural (bukan kamus slang kaku yang bisa merusak konteks medis/finansial/teknis).
3. **Multi-Column Text Fusion**: Mampu menggabungkan lebih dari satu kolom teks (misal `Judul` + `Komentar`) dengan penanda struktural.
4. **Adaptive Task Resolver**: Menangani otomatis apakah sentimen bertipe Biner (Positif/Negatif), Multi-Kelas (Positif/Netral/Negatif), atau Regresi Kontinu (Rating/Skor 1.0 - 5.0).
5. **Dual Engine**: Baseline Cepat (TF-IDF + Balanced GBDT/Linear) dan Transformer (IndoBERT/RoBERTa) dengan weighted loss.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_squared_error, classification_report
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
print("Environment siap. Seed:", SEED)


## 1. Unified Configuration Card
Cukup atur konfigurasi pada sel di bawah ini. Jika kolom dibiarkan `None`, sistem akan **mendeteksi secara otomatis** dari data yang diberikan.


In [ ]:
CONFIG = {
    # Jalur berkas dataset kompetisi (CSV, TSV, Parquet, atau XLSX)
    "train_path": "train.csv",
    "test_path": "test.csv", # Isi None jika tidak ada test set terpisah
    
    # Kolom: Set None untuk auto-detect, atau masukkan string nama kolom
    # Bisa berupa list jika ingin menggabungkan beberapa kolom teks, misal ["title", "review"]
    "text_cols": None, 
    "target_col": None,
    "id_col": None,
    
    # Preprocessing toggles (Domain-Agnostic)
    "lower": True,
    "strip_html": True,
    "strip_urls": True,
    "normalize_repeated_chars": True, # 'bangeeeet' -> 'banget', 'soooo' -> 'so'
    "prefix_tags": True, # Jika >1 kolom teks, beri tag [KOLOM_A] ... [KOLOM_B]
    
    # Modeling parameters
    "max_features": 15000,
    "n_splits": 5
}

print("Konfigurasi tersimpan.")


## 2. Auto-Discovery & Data Ingestion Engine
Membaca data dan otomatis memetakan skema tanpa asumsi nama kolom tertentu.


In [ ]:
def load_and_inspect_data(cfg):
    train_p = cfg["train_path"]
    test_p = cfg["test_path"]
    
    # Fallback generator data demo jika berkas kompetisi belum ditaruh
    if not os.path.exists(train_p):
        print(f"Peringatan: '{train_p}' belum ditemukan. Menjalankan fallback generator simulasi...")
        train_df = pd.DataFrame({
            "ulasan_id": [f"ID_{i:03d}" for i in range(1, 13)],
            "judul_ulasan": [
                "Layanan mantap", "Sistem error parah", "Biasa saja", "Sangat puas",
                "Kotor dan bau", "Sesuai jadwal", "Internet mati terus", "Sangat membantu",
                "Harga murah kualitas oke", "Tagihan keliru bikin pusing", "Informasi standar", "Selesai cepat"
            ],
            "isi_ulasan": [
                "Petugas loket sangat ramah bgt dan cepat layani antrean.",
                "Portal sering down pas pengisian data penting, kecewa berat!!",
                "Administrasi berjalan normal tanpa kendala yang berarti.",
                "Fasilitas lengkap dan materi perkuliahan sangat jelas.",
                "Toilet di gedung D rusak dan tidak ada air, tolong diperbaiki min.",
                "Kegiatan berjalan sesuai jadwal di kalender akademik.",
                "Bayar mahal tapi wifi lemot minta ampun ga bisa dipake.",
                "Beasiswa cair tepat waktu sangat meringankan biaya hidup.",
                "Kantin bersih makanan enak dan porsi memuaskan.",
                "Status di akun belum lunas padahal resi transfer sudah ada.",
                "Pengumuman disampaikan melalui saluran resmi fakultas.",
                "Proses legalisir dokumen hanya butuh waktu 1 hari."
            ],
            "label_sentimen": [
                "positif", "negatif", "netral", "positif", "negatif", "netral",
                "negatif", "positif", "positif", "negatif", "netral", "positif"
            ]
        })
        test_df = pd.DataFrame({
            "ulasan_id": ["ID_101", "ID_102"],
            "judul_ulasan": ["Bagus sekali", "Kecewa pelayanan"],
            "isi_ulasan": ["Semua staf bekerja profesional dan cepat.", "Respon CS sangat lambat dan berbelit-belit."]
        })
    else:
        # Dukungan multi-format (CSV, TSV, XLSX, Parquet)
        ext = os.path.splitext(train_p)[1].lower()
        if ext in ['.xlsx', '.xls']:
            train_df = pd.read_excel(train_p)
            test_df = pd.read_excel(test_p) if test_p and os.path.exists(test_p) else None
        elif ext == '.parquet':
            train_df = pd.read_parquet(train_p)
            test_df = pd.read_parquet(test_p) if test_p and os.path.exists(test_p) else None
        else:
            train_df = pd.read_csv(train_p)
            test_df = pd.read_csv(test_p) if test_p and os.path.exists(test_p) else None

    # Auto-detect ID Column
    id_col = cfg["id_col"]
    if not id_col or id_col not in train_df.columns:
        id_candidates = [c for c in train_df.columns if re.search(r"id|uuid|kode|no", c, re.IGNORECASE)]
        id_col = id_candidates[0] if id_candidates else train_df.columns[0]

    # Auto-detect Target Column (Kolom dengan kardinalitas relatif kecil tapi bukan ID)
    target_col = cfg["target_col"]
    if not target_col or target_col not in train_df.columns:
        target_candidates = [c for c in train_df.columns if c != id_col and train_df[c].nunique() < 30 and train_df[c].dtype != float]
        if not target_candidates:
            target_candidates = [c for c in train_df.columns if c != id_col and re.search(r"target|label|sentimen|sentiment|class|kategori", c, re.IGNORECASE)]
        target_col = target_candidates[0] if target_candidates else train_df.columns[-1]

    # Auto-detect Text Column(s)
    text_cols = cfg["text_cols"]
    if not text_cols:
        string_cols = [c for c in train_df.columns if c not in [id_col, target_col] and (train_df[c].dtype == object or pd.api.types.is_string_dtype(train_df[c]))]
        if len(string_cols) == 1:
            text_cols = string_cols
        elif len(string_cols) > 1:
            # Urutkan berdasarkan rata-rata panjang teks (paling panjang pertama)
            avg_lens = {c: train_df[c].astype(str).str.len().mean() for c in string_cols}
            sorted_cols = sorted(avg_lens, key=avg_lens.get, reverse=True)
            text_cols = sorted_cols # Gabungkan seluruh kolom string yang informatif!
        else:
            text_cols = [train_df.columns[1]]
    elif isinstance(text_cols, str):
        text_cols = [text_cols]

    return train_df, test_df, id_col, target_col, text_cols

train_df, test_df, ID_COL, TARGET_COL, TEXT_COLS = load_and_inspect_data(CONFIG)
print(f"=== HASIL DETEKSI SKEMA DATASET ===")
print(f"Kolom ID      : '{ID_COL}'")
print(f"Kolom Target  : '{TARGET_COL}' (Total {train_df[TARGET_COL].nunique()} Kelas)")
print(f"Kolom Teks    : {TEXT_COLS}")
print(f"Ukuran Train  : {train_df.shape}")
if test_df is not None:
    print(f"Ukuran Test   : {test_df.shape}")
print(f"\nDistribusi Target:\n{train_df[TARGET_COL].value_counts()}")


## 3. Domain-Agnostic Modular Text Cleaner & Field Fusion
Menggabungkan beberapa kolom teks (jika ada) dan membersihkan format teks secara modular tanpa memaksakan kamus kata tertentu.


In [ ]:
class UniversalTextTransformer:
    def __init__(self, text_cols, lower=True, strip_html=True, strip_urls=True, normalize_repeat=True, prefix_tags=True):
        self.text_cols = text_cols
        self.lower = lower
        self.strip_html = strip_html
        self.strip_urls = strip_urls
        self.normalize_repeat = normalize_repeat
        self.prefix_tags = prefix_tags

    def clean_single_string(self, text):
        if not isinstance(text, str):
            return ""
        s = text
        if self.strip_html:
            s = re.sub(r"<[^>]+>", " ", s)
        if self.strip_urls:
            s = re.sub(r"https?://\S+|www\.\S+", " ", s)
        if self.normalize_repeat:
            # Memotong huruf berulang lebih dari 2 kali: 'bangeeeet' -> 'banget', 'waaaww' -> 'waw'
            s = re.sub(r"(.)\1{2,}", r"\1", s)
        if self.lower:
            s = s.lower()
        s = re.sub(r"\s+", " ", s).strip()
        return s

    def transform(self, df):
        cleaned_series = []
        for c in self.text_cols:
            col_cleaned = df[c].fillna("").astype(str).apply(self.clean_single_string)
            if self.prefix_tags and len(self.text_cols) > 1:
                tag_name = re.sub(r"[^\w]", "", c).upper()
                col_cleaned = f"[{tag_name}] " + col_cleaned
            cleaned_series.append(col_cleaned)

        if len(cleaned_series) == 1:
            return cleaned_series[0]
        else:
            # Field fusion: Gabungkan seluruh kolom teks dengan spasi
            fused = cleaned_series[0]
            for s in cleaned_series[1:]:
                fused = fused + " " + s
            return fused.str.strip()

transformer = UniversalTextTransformer(
    text_cols=TEXT_COLS,
    lower=CONFIG["lower"],
    strip_html=CONFIG["strip_html"],
    strip_urls=CONFIG["strip_urls"],
    normalize_repeat=CONFIG["normalize_repeated_chars"],
    prefix_tags=CONFIG["prefix_tags"]
)

train_df["fused_text"] = transformer.transform(train_df)
if test_df is not None:
    test_df["fused_text"] = transformer.transform(test_df)

print("Contoh Teks Terpadu:")
for i in range(min(2, len(train_df))):
    print(f"[{i}] {train_df['fused_text'].iloc[i][:150]}...")


## 4. Ekstraksi Fitur Teks & Stilometri Struktural
Mengekstrak sinyal teks menggunakan Sublinear TF-IDF (Word N-Gram + Char-wb) dan fitur morfometrik bahasa.


In [ ]:
# 1. Fitur Stilometri Struktural (Bekerja di semua bahasa & domain)
def extract_structural_stylometrics(df, col):
    feats = pd.DataFrame(index=df.index)
    s = df[col].astype(str)
    feats["char_len"] = s.str.len()
    feats["word_count"] = s.apply(lambda x: len(x.split()))
    feats["avg_word_len"] = feats["char_len"] / (feats["word_count"] + 1)
    feats["caps_ratio"] = s.apply(lambda x: sum(1 for c in x if c.isupper())) / (feats["char_len"] + 1)
    feats["digit_ratio"] = s.apply(lambda x: sum(1 for c in x if c.isdigit())) / (feats["char_len"] + 1)
    feats["exclam_count"] = s.apply(lambda x: x.count('!'))
    feats["question_count"] = s.apply(lambda x: x.count('?'))
    feats["punct_count"] = s.apply(lambda x: len(re.findall(r"[^\w\s]", x)))
    return feats

style_tr = extract_structural_stylometrics(train_df, "fused_text")
style_te = extract_structural_stylometrics(test_df, "fused_text") if test_df is not None else None

# 2. Dual-Engine TF-IDF: Word (1, 2) + Char-WB (3, 5)
# Char-wb (character within word boundaries) menangkap variasi slang/typo tanpa hardcoding kamus!
word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=CONFIG["max_features"], sublinear_tf=True)
char_vec = TfidfVectorizer(ngram_range=(3, 5), analyzer="char_wb", max_features=CONFIG["max_features"], sublinear_tf=True)

X_tr_w = word_vec.fit_transform(train_df["fused_text"])
X_tr_c = char_vec.fit_transform(train_df["fused_text"])
X_train = hstack([X_tr_w, X_tr_c, csr_matrix(style_tr.values)]).tocsr()

if test_df is not None:
    X_te_w = word_vec.transform(test_df["fused_text"])
    X_te_c = char_vec.transform(test_df["fused_text"])
    X_test = hstack([X_te_w, X_te_c, csr_matrix(style_te.values)]).tocsr()
else:
    X_test = None

print(f"Dimensi Matriks Fitur Lengkap: {X_train.shape}")


## 5. Pelatihan Model & Validasi Silang Adaptif (Biner vs Multi-Kelas)


In [ ]:
# Encoding Target
is_regression = pd.api.types.is_numeric_dtype(train_df[TARGET_COL]) and train_df[TARGET_COL].nunique() > 20

if not is_regression:
    labels = sorted(train_df[TARGET_COL].unique())
    num_classes = len(labels)
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for i, l in enumerate(labels)}
    y_train = train_df[TARGET_COL].map(label2id).values
    task_mode = "binary" if num_classes == 2 else "multiclass"
else:
    y_train = train_df[TARGET_COL].values
    task_mode = "regression"
    labels, label2id, id2label, num_classes = None, None, None, 1

print(f"Task Mode Teridentifikasi: '{task_mode.upper()}' (Kelas: {num_classes})")

# Validasi Silang
n_splits = min(CONFIG["n_splits"], min(train_df[TARGET_COL].value_counts()) if not is_regression else 5)
n_splits = max(2, n_splits)

if not is_regression:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_preds = np.zeros((len(train_df), num_classes)) if num_classes > 2 else np.zeros(len(train_df))
else:
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_preds = np.zeros(len(train_df))

test_preds = []

print(f"Menjalankan {n_splits}-Fold Cross-Validation...")
for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train if not is_regression else None)):
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_va, y_va = X_train[va_idx], y_train[va_idx]
    
    if task_mode == "regression":
        model = Ridge(alpha=1.0, random_state=SEED + fold)
        model.fit(X_tr, y_tr)
        preds_va = model.predict(X_va)
        oof_preds[va_idx] = preds_va
        score = mean_squared_error(y_va, preds_va, squared=False)
        print(f"Fold {fold+1} | RMSE: {score:.4f}")
        if X_test is not None: test_preds.append(model.predict(X_test))
    elif task_mode == "binary":
        model = LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=SEED + fold)
        model.fit(X_tr, y_tr)
        probas = model.predict_proba(X_va)[:, 1]
        oof_preds[va_idx] = probas
        auc = roc_auc_score(y_va, probas)
        f1 = f1_score(y_va, (probas >= 0.5).astype(int), zero_division=0)
        print(f"Fold {fold+1} | ROC-AUC: {auc:.4f} | F1: {f1:.4f}")
        if X_test is not None: test_preds.append(model.predict_proba(X_test)[:, 1])
    else: # multiclass
        model = LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=SEED + fold)
        model.fit(X_tr, y_tr)
        probas = model.predict_proba(X_va)
        oof_preds[va_idx] = probas
        preds = np.argmax(probas, axis=1)
        acc = accuracy_score(y_va, preds)
        mf1 = f1_score(y_va, preds, average="macro", zero_division=0)
        print(f"Fold {fold+1} | Akurasi: {acc*100:.2f}% | Macro-F1: {mf1*100:.2f}%")
        if X_test is not None: test_preds.append(model.predict_proba(X_test))


## 6. Laporan Kinerja OOF & Ekspor Submission


In [ ]:
print("=== EVALUASI AKHIR OUT-OF-FOLD (OOF) ===")
if task_mode == "multiclass":
    final_pred_classes = np.argmax(oof_preds, axis=1)
    print(f"Overall Accuracy : {accuracy_score(y_train, final_pred_classes)*100:.2f}%")
    print(f"Overall Macro-F1 : {f1_score(y_train, final_pred_classes, average='macro', zero_division=0)*100:.2f}%")
    print("\nLaporan Klasifikasi:\n", classification_report(y_train, final_pred_classes, target_names=labels, zero_division=0))
elif task_mode == "binary":
    # Optimasi Ambang Batas Threshold
    thresholds = np.linspace(0.1, 0.9, 81)
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        sc = f1_score(y_train, (oof_preds >= t).astype(int), zero_division=0)
        if sc > best_f1: best_f1, best_t = sc, t
    print(f"ROC-AUC: {roc_auc_score(y_train, oof_preds):.4f}")
    print(f"Best Threshold ({best_t:.2f}) -> F1: {best_f1:.4f} (Default 0.5: {f1_score(y_train, (oof_preds>=0.5).astype(int)):.4f})")
else:
    print(f"Overall RMSE: {mean_squared_error(y_train, oof_preds, squared=False):.4f}")

# Ekspor Berkas Submission jika ada Test Set
if test_df is not None and len(test_preds) > 0:
    avg_preds = np.mean(test_preds, axis=0)
    if task_mode == "multiclass":
        out_labels = [id2label[i] for i in np.argmax(avg_preds, axis=1)]
    elif task_mode == "binary":
        out_labels = (avg_preds >= best_t).astype(int)
    else:
        out_labels = avg_preds
    
    sub = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET_COL: out_labels
    })
    sub.to_csv("submission_sentiment_universal.csv", index=False)
    print("\nBerkas 'submission_sentiment_universal.csv' berhasil disimpan:")
    print(sub.head())
